## Setting Up

In [1]:
# IMPORTS
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import ipyvolume as ipv
import neuropythy as ny

import torch
import torch.nn.functional as F
import torch.optim as optim

from torch_geometric.utils import to_undirected
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.nn import SplineConv

libpath = '/home/achegu/repos/mesh-annot/src'
if libpath not in sys.path:
    sys.path.append(libpath)

import mesh_annot

/home/achegu/.conda/envs/mesh-annot-env/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /home/achegu/.conda/envs/mesh-annot-env/lib/python3.12/site-packages/torch_scatter/_version_cpu.so
  import torch_geometric.typing
/home/achegu/.conda/envs/mesh-annot-env/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /home/achegu/.conda/envs/mesh-annot-env/lib/python3.12/site-packages/torch_sparse/_version_cpu.so
  import torch_geometric.typing


In [2]:
# CONFIGURE FREESURFER HOME
ny.config['freesurfer_home'] = '/opt/freesurfer/'
ds = ny.config['freesurfer_home']

In [3]:
%matplotlib inline

## FreeSurfer Stuff

In [ ]:
sub = ny.freesurfer_subject('nben')
fsa = ny.freesurfer_subject('fsaverage')
interpolated = sub.lh.interpolate(fsa.lh, sub.lh.white_surface.coordinates) # A (B, A)
surface = fsa.lh.white_surface.copy(coordinates=interpolated)

In [ ]:
surface

In [ ]:
ny.cortex_plot(surface)

In [ ]:
ny.data['hcp_lines'].subject_list

In [ ]:
sub = ny.hcp_subject('100610')
fsa = ny.freesurfer_subject('fsaverage')
interpolated = sub.lh.interpolate(fsa.lh, sub.lh.white_surface.coordinates) # A (B, A)
surface = fsa.lh.white_surface.copy(coordinates=interpolated)

In [ ]:
# TRY THIS NEXT
sub = ny.hcp_subject('100610')
fsa = ny.freesurfer_subject('fsaverage')
interpolated = sub.lh.interpolate(fsa.lh, 'curvature') # A (B, A)
# surface = fsa.lh.prop('curvature').with_prop(curvature=interpolated)

In [ ]:
properties = {
    'prf_x':'prf_x', 
    'prf_y':'prf_y', 
    'prf_variance_explained':'prf_cod', 
    'prf_radius':'prf_sigma'
}
sids = ny.data['hcp_lines'].subject_list
cache_dir = '/data/visual-autolabel/surface'

for sid in sids:
    print(sid)
    for prop in properties.keys():
        print(prop)
        ny = ny.reload_neuropythy()
        fsa = ny.freesurfer_subject('fsaverage')
        sub = ny.hcp_subject(sid) # Just pass an integer
        interpolated = sub.lh.interpolate(fsa.lh, prop)

        filepath = os.path.join(cache_dir, properties.get(prop), f'{sid}.lh.mgz')
        if not os.path.exists(filepath):
            print(sid)
            ny.save(filepath, interpolated)
            print(f"Saving out to: {filepath}")

100610
prf_x
prf_y
prf_variance_explained
100610
Saving out to: /data/visual-autolabel/surface/prf_cod/100610.lh.mgz
prf_radius
100610
Saving out to: /data/visual-autolabel/surface/prf_sigma/100610.lh.mgz
118225
prf_x
118225
Saving out to: /data/visual-autolabel/surface/prf_x/118225.lh.mgz
prf_y
118225
Saving out to: /data/visual-autolabel/surface/prf_y/118225.lh.mgz
prf_variance_explained
118225
Saving out to: /data/visual-autolabel/surface/prf_cod/118225.lh.mgz
prf_radius
118225
Saving out to: /data/visual-autolabel/surface/prf_sigma/118225.lh.mgz
140117
prf_x
140117
Saving out to: /data/visual-autolabel/surface/prf_x/140117.lh.mgz
prf_y
140117
Saving out to: /data/visual-autolabel/surface/prf_y/140117.lh.mgz
prf_variance_explained
140117
Saving out to: /data/visual-autolabel/surface/prf_cod/140117.lh.mgz
prf_radius
140117
Saving out to: /data/visual-autolabel/surface/prf_sigma/140117.lh.mgz
158136
prf_x
158136
Saving out to: /data/visual-autolabel/surface/prf_x/158136.lh.mgz
prf_y
1

In [12]:
sids[1]

100610

In [6]:
fsa.lh.registrations

lmap({'benson17_retinotopy': <lazy>, 'fsaverage': Mesh(<3D>, <327680 faces>, <163842 vertices>), 'benson17': <lazy>, 'benson14': <lazy>, 'fsaverage_sym': <lazy>, 'native': <lazy>, 'benson14_retinotopy': <lazy>})

In [11]:
fsa.path

'/data/freesurfer/subjects/fsaverage'

In [ ]:
sids = ny.data['hcp_lines'].subject_list
cache_dir = '/data/visual-autolabel/surface'

for sid in sids:
        ny = ny.reload_neuropythy()
        sub = ny.hcp_subject(f'{sid}')
        fsa = ny.freesurfer_subject('fsaverage')
        interpolated = sub.rh.interpolate(fsa.rh, sub.rh.white_surface.coordinates)
        surface = fsa.rh.white_surface.copy(coordinates=interpolated)
    
        # Use .mgz when we start saving out the properties
        filepath = os.path.join(cache_dir, f'{sid}.rh.mesh')
        if not os.path.exists(filepath):
            print(f"Saving out subject: {sid}")
            ny.save(filepath, surface, 'freesurfer_geometry')